# AI Financial Analyst Agent

A comprehensive AI-powered financial analysis system built with:
- **LangGraph** for multi-agent orchestration
- **Groq (Llama 3)** for fast LLM inference
- **yfinance** for real-time market data
- **ChromaDB** for RAG on earnings calls
- **Streamlit** for the web interface

---

## Phase 1: Foundation & Data Engineering 

In [ ]:
# Install dependencies
!pip install yfinance pandas plotly streamlit langchain langchain-groq langgraph chromadb pypdf2 duckduckgo-search sentence-transformers langchain-community -q

In [ ]:
# Core imports
import os
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, TypedDict, Annotated, Literal
import operator

import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from getpass import getpass

warnings.filterwarnings('ignore')

# Configure Groq API Key (get a free key at https://console.groq.com)
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API Key: ')

print('✅ Environment & core imports initialized!')

### Week 1: Data Ingestion Pipeline
Building the core `MarketData` class for fetching historical OHLCV stock data and company fundamentals via `yfinance`, real-time financial news via DuckDuckGo Search, and interactive candlestick charts with Plotly.

In [ ]:
class MarketData:
    """Handles fetching and formatting stock market data and fundamentals from Yahoo Finance."""

    def get_stock_history(
        self,
        ticker: str,
        days: int = 30,
        as_of_date: Optional[str] = None
    ) -> pd.DataFrame:
        """Fetch historical OHLCV data for a given ticker over the past N days."""
        end_dt = datetime.strptime(as_of_date, "%Y-%m-%d") if as_of_date else datetime.now()
        start_dt = end_dt - timedelta(days=days)

        stock = yf.Ticker(ticker.upper())
        df = stock.history(
            start=start_dt.strftime("%Y-%m-%d"),
            end=end_dt.strftime("%Y-%m-%d")
        )
        if df.empty:
            raise ValueError(f"No historical price data found for ticker '{ticker}'.")
        return df

    def get_fundamentals(self, ticker: str) -> Dict[str, Any]:
        """Fetch key fundamental metrics for a given ticker."""
        stock = yf.Ticker(ticker.upper())
        info = stock.info
        return {
            "ticker": ticker.upper(),
            "name": info.get("longName", ticker.upper()),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "current_price": info.get("currentPrice", info.get("regularMarketPrice", "N/A")),
            "market_cap": info.get("marketCap", 0) or 0,
            "pe_ratio": info.get("trailingPE", "N/A"),
            "forward_pe": info.get("forwardPE", "N/A"),
            "eps": info.get("trailingEps", "N/A"),
            "dividend_yield": info.get("dividendYield", "N/A"),
            "52_week_high": info.get("fiftyTwoWeekHigh", "N/A"),
            "52_week_low": info.get("fiftyTwoWeekLow", "N/A"),
            "recommendation": info.get("recommendationKey", "N/A"),
        }

    @staticmethod
    def format_market_cap(value: float) -> str:
        """Format large market cap numbers into human-readable strings (T, B, M)."""
        if not isinstance(value, (int, float)) or value <= 0:
            return "N/A"
        for divisor, suffix in ((1e12, "T"), (1e9, "B"), (1e6, "M")):
            if value >= divisor:
                return f"${value / divisor:.2f}{suffix}"
        return f"${value:,.0f}"


def fetch_financial_news(ticker: str, max_results: int = 5) -> List[Dict[str, str]]:
    """Fetch recent financial news articles for a stock ticker using DuckDuckGo."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.news(f"{ticker} stock financial news", max_results=max_results))
        news_items = [
            {
                "title": r.get("title", "No title"),
                "source": r.get("source", "Unknown"),
                "date": r.get("date", ""),
                "link": r.get("url", ""),
                "snippet": (r.get("body") or "")[:250],
            }
            for r in results
        ]
        return news_items or [{"title": f"No recent news found for {ticker}", "source": "", "date": "", "link": "", "snippet": ""}]
    except Exception as e:
        return [{"title": f"News fetch unavailable: {e}", "source": "", "date": "", "link": "", "snippet": ""}]


def create_candlestick_chart(df: pd.DataFrame, ticker: str) -> go.Figure:
    """Create an interactive Plotly candlestick chart with 20/50-day SMA and volume subplot."""
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.72, 0.28],
        subplot_titles=(f"{ticker.upper()} Price & Moving Averages", "Volume")
    )

    # Candlestick trace
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name="OHLC"
        ),
        row=1, col=1
    )

    # Moving averages
    if len(df) >= 20:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["Close"].rolling(20).mean(),
                name="SMA 20",
                line=dict(color="#FFD700", width=1.5)
            ),
            row=1, col=1
        )
    if len(df) >= 50:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["Close"].rolling(50).mean(),
                name="SMA 50",
                line=dict(color="#00BFFF", width=1.5)
            ),
            row=1, col=1
        )

    # Volume bars colored by daily price direction
    colors = ["#00C805" if c >= o else "#FF3B30" for o, c in zip(df["Open"], df["Close"])]
    fig.add_trace(
        go.Bar(x=df.index, y=df["Volume"], marker_color=colors, name="Volume"),
        row=2, col=1
    )

    fig.update_layout(
        template="plotly_dark",
        xaxis_rangeslider_visible=False,
        height=550,
        margin=dict(l=40, r=40, t=50, b=40)
    )
    return fig

In [ ]:
# Test Week 1 Data Pipeline on AAPL
market_data = MarketData()
fundamentals = market_data.get_fundamentals("AAPL")

print(f"📊 {fundamentals['name']} ({fundamentals['ticker']})")
print(f"   Price: ${fundamentals['current_price']} | Market Cap: {market_data.format_market_cap(fundamentals['market_cap'])}")
print(f"   P/E Ratio: {fundamentals['pe_ratio']} | 52W Range: ${fundamentals['52_week_low']} - ${fundamentals['52_week_high']}")
print(f"   Analyst Consensus: {fundamentals['recommendation'].upper()}\n")

print("📰 Latest News Headlines:")
for idx, article in enumerate(fetch_financial_news("AAPL", max_results=3), 1):
    print(f"   {idx}. {article['title']} ({article['source']})")

hist_df = market_data.get_stock_history("AAPL", days=90)
fig = create_candlestick_chart(hist_df, "AAPL")
fig.show()

### Week 2: LLM Integration & Basic UI
Connecting Groq's ultra-fast Llama 3 inference engine via `langchain-groq` to analyze structured financial data.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage


def get_llm(model_name: str = "llama-3.1-8b-instant", temperature: float = 0.1) -> ChatGroq:
    """Initialize the Groq ChatLLM instance."""
    return ChatGroq(model=model_name, temperature=temperature)


def basic_financial_summary(ticker: str) -> str:
    """Fetch fundamentals and recent history, then generate a concise LLM summary."""
    llm = get_llm()
    f = market_data.get_fundamentals(ticker)
    df = market_data.get_stock_history(ticker, days=30)
    pct_change = ((df["Close"].iloc[-1] - df["Close"].iloc[0]) / df["Close"].iloc[0]) * 100

    prompt = (
        f"Company: {f['name']} ({f['ticker']})\n"
        f"Sector: {f['sector']} | Industry: {f['industry']}\n"
        f"Current Price: ${f['current_price']} | 30-Day Change: {pct_change:+.2f}%\n"
        f"Market Cap: {market_data.format_market_cap(f['market_cap'])} | P/E: {f['pe_ratio']} | Forward P/E: {f['forward_pe']}\n"
        f"52-Week Range: ${f['52_week_low']} - ${f['52_week_high']}\n\n"
        "Provide a concise 3-bullet financial snapshot based strictly on the metrics above."
    )
    response = llm.invoke([
        SystemMessage(content="You are an expert Wall Street financial analyst. Never fabricate numbers."),
        HumanMessage(content=prompt)
    ])
    return response.content


print(basic_financial_summary("AAPL"))

---
## Phase 2: Agent Orchestration

### Week 3: Tool Definition & Single ReAct Agent
Wrapping our market data, historical performance, and news functions as LangChain `@tool` functions and orchestrating a tool-calling agent in **LangGraph**.

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


@tool
def get_stock_price(ticker: str) -> str:
    """Get current stock price, valuation ratios, and key fundamentals for a ticker symbol."""
    try:
        f = market_data.get_fundamentals(ticker)
        return (
            f"{f['name']} ({f['ticker']})\n"
            f"Current Price: ${f['current_price']}\n"
            f"Market Cap: {market_data.format_market_cap(f['market_cap'])}\n"
            f"Trailing P/E: {f['pe_ratio']} | Forward P/E: {f['forward_pe']} | EPS: {f['eps']}\n"
            f"52-Week High: ${f['52_week_high']} | 52-Week Low: ${f['52_week_low']}\n"
            f"Analyst Recommendation: {f['recommendation']}"
        )
    except Exception as e:
        return f"Error fetching fundamentals for {ticker}: {e}"


@tool
def get_stock_history_tool(ticker: str, days: int = 30) -> str:
    """Get historical stock price performance and volatility metrics over the past N days."""
    try:
        df = market_data.get_stock_history(ticker, days=days)
        start_p, end_p = df["Close"].iloc[0], df["Close"].iloc[-1]
        pct = ((end_p - start_p) / start_p) * 100
        return (
            f"{ticker.upper()} ({days}-Day Performance):\n"
            f"Start Close: ${start_p:.2f} -> Latest Close: ${end_p:.2f} ({pct:+.2f}%)\n"
            f"Period High: ${df['High'].max():.2f} | Period Low: ${df['Low'].min():.2f}\n"
            f"Average Daily Volume: {df['Volume'].mean():,.0f}"
        )
    except Exception as e:
        return f"Error fetching history for {ticker}: {e}"


@tool
def get_stock_news(ticker: str) -> str:
    """Get the latest financial news headlines and snippets for a stock ticker."""
    news = fetch_financial_news(ticker, max_results=5)
    lines = [
        f"{i}. {n['title']} ({n['source']}) - {n['snippet']}"
        for i, n in enumerate(news, 1)
    ]
    return "\n".join(lines)


FINANCIAL_TOOLS = [get_stock_price, get_stock_history_tool, get_stock_news]


class SingleAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]


def create_single_agent():
    """Build a single tool-calling LangGraph financial analyst agent."""
    llm_with_tools = get_llm().bind_tools(FINANCIAL_TOOLS)
    sys_prompt = (
        "You are an AI Financial Analyst. Always use the provided tools to retrieve real-time "
        "market data, historical trends, and news before answering. Never invent financial numbers."
    )

    def agent_node(state: SingleAgentState) -> Dict[str, Any]:
        messages = [SystemMessage(content=sys_prompt)] + state["messages"]
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def should_continue(state: SingleAgentState) -> Literal["tools", "end"]:
        last_msg = state["messages"][-1]
        return "tools" if getattr(last_msg, "tool_calls", None) else "end"

    workflow = StateGraph(SingleAgentState)
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", ToolNode(FINANCIAL_TOOLS))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    workflow.add_edge("tools", "agent")
    return workflow.compile()


single_agent = create_single_agent()
result = single_agent.invoke({
    "messages": [HumanMessage(content="What is Microsoft's (MSFT) current valuation and 30-day price trend?")]
})
print(result["messages"][-1].content)

### Week 4: Multi-Agent Architecture
Scaling from a single agent to a specialized multi-agent team orchestrated via **LangGraph**:
1. **Router Agent**: Extracts the target ticker symbol and intent from the user query.
2. **Market Data Agent**: Analyzes valuation multiples, fundamentals, and technical price action.
3. **News Sentiment Agent**: Evaluates recent news catalysts, headlines, and market sentiment.
4. **Head Analyst Agent**: Synthesizes the quantitative and qualitative reports into an executive investment memo.

In [ ]:
class MultiAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    query: str
    ticker: str
    market_analysis: str
    news_analysis: str
    final_report: str


def create_multi_agent_system():
    """Build the multi-agent financial research graph."""
    llm = get_llm()

    def router_node(state: MultiAgentState) -> Dict[str, Any]:
        prompt = (
            f"Extract ONLY the stock ticker symbol (e.g. AAPL, NVDA, TSLA) from this query: '{state['query']}'. "
            "Return strictly the uppercase ticker symbol and nothing else."
        )
        raw = llm.invoke([HumanMessage(content=prompt)]).content.strip().upper()
        ticker = raw.split()[0].strip(".,'\"")
        return {"ticker": ticker}

    def market_analyst_node(state: MultiAgentState) -> Dict[str, Any]:
        ticker = state["ticker"]
        fundamentals_str = get_stock_price.invoke({"ticker": ticker})
        history_str = get_stock_history_tool.invoke({"ticker": ticker, "days": 60})
        prompt = (
            f"Ticker: {ticker}\n\nFundamentals:\n{fundamentals_str}\n\n"
            f"Price Action:\n{history_str}\n\n"
            "Provide a structured quantitative analysis covering valuation, momentum, and key price levels."
        )
        res = llm.invoke([
            SystemMessage(content="You are the Quantitative Market Data Analyst on an investment committee."),
            HumanMessage(content=prompt)
        ])
        return {"market_analysis": res.content}

    def news_analyst_node(state: MultiAgentState) -> Dict[str, Any]:
        ticker = state["ticker"]
        news_str = get_stock_news.invoke({"ticker": ticker})
        prompt = (
            f"Ticker: {ticker}\n\nRecent News:\n{news_str}\n\n"
            "Classify overall sentiment (Bullish / Neutral / Bearish) and summarize the top catalysts and risks."
        )
        res = llm.invoke([
            SystemMessage(content="You are the Financial News & Sentiment Analyst on an investment committee."),
            HumanMessage(content=prompt)
        ])
        return {"news_analysis": res.content}

    def head_analyst_node(state: MultiAgentState) -> Dict[str, Any]:
        prompt = (
            f"User Query: {state['query']} (Ticker: {state['ticker']})\n\n"
            f"--- QUANTITATIVE MARKET ANALYSIS ---\n{state['market_analysis']}\n\n"
            f"--- NEWS & SENTIMENT ANALYSIS ---\n{state['news_analysis']}\n\n"
            "Synthesize these findings into a comprehensive Investment Report with:\n"
            "1. Executive Summary\n"
            "2. Fundamental & Technical Outlook\n"
            "3. News Catalysts & Risks\n"
            "4. Final Recommendation (BUY / HOLD / SELL) with justification."
        )
        res = llm.invoke([
            SystemMessage(content="You are the Head Portfolio Strategist. Be rigorous, balanced, and data-driven."),
            HumanMessage(content=prompt)
        ])
        return {"final_report": res.content, "messages": [AIMessage(content=res.content)]}

    graph = StateGraph(MultiAgentState)
    graph.add_node("router", router_node)
    graph.add_node("market_analyst", market_analyst_node)
    graph.add_node("news_analyst", news_analyst_node)
    graph.add_node("head_analyst", head_analyst_node)

    graph.set_entry_point("router")
    graph.add_edge("router", "market_analyst")
    graph.add_edge("market_analyst", "news_analyst")
    graph.add_edge("news_analyst", "head_analyst")
    graph.add_edge("head_analyst", END)

    return graph.compile()


multi_agent_system = create_multi_agent_system()
ma_output = multi_agent_system.invoke({
    "messages": [],
    "query": "Perform a full investment analysis on NVIDIA (NVDA).",
    "ticker": "",
    "market_analysis": "",
    "news_analysis": "",
    "final_report": ""
})
print(ma_output["final_report"])

### Week 5: RAG for Earnings Calls (ChromaDB)
Building a Retrieval-Augmented Generation (RAG) pipeline using **ChromaDB**, **PyPDF2**, and **HuggingFace Sentence Transformers** (`all-MiniLM-L6-v2`) to index and query earnings call transcripts and SEC filings.

In [ ]:
import chromadb
from PyPDF2 import PdfReader
from langchain_community.embeddings import HuggingFaceEmbeddings


class EarningsCallRAG:
    """RAG system for indexing and querying earnings call transcripts and financial PDFs."""

    def __init__(self, collection_name: str = "earnings_calls"):
        self.embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def _chunk_text(self, text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
        chunks = []
        step = max(chunk_size - overlap, 1)
        for i in range(0, len(text), step):
            chunk = text[i : i + chunk_size].strip()
            if chunk:
                chunks.append(chunk)
        return chunks

    def ingest_text(self, text: str, source_id: str, chunk_size: int = 800, overlap: int = 150) -> int:
        """Chunk and index raw transcript text into ChromaDB."""
        chunks = self._chunk_text(text, chunk_size=chunk_size, overlap=overlap)
        embeddings = self.embedder.embed_documents(chunks)
        ids = [f"{source_id}_chunk_{i}" for i in range(len(chunks))]
        metadatas = [{"source": source_id, "chunk_index": i} for i in range(len(chunks))]
        self.collection.upsert(documents=chunks, embeddings=embeddings, ids=ids, metadatas=metadatas)
        return len(chunks)

    def ingest_pdf(self, pdf_path: str, chunk_size: int = 800, overlap: int = 150) -> int:
        """Extract text from a PDF earnings release/transcript and index it into ChromaDB."""
        reader = PdfReader(pdf_path)
        full_text = "\n".join((page.extract_text() or "") for page in reader.pages)
        return self.ingest_text(full_text, source_id=os.path.basename(pdf_path), chunk_size=chunk_size, overlap=overlap)

    def search(self, query: str, k: int = 3) -> List[str]:
        """Retrieve the top-k most relevant transcript passages for a query."""
        if self.collection.count() == 0:
            return []
        q_emb = self.embedder.embed_query(query)
        results = self.collection.query(query_embeddings=[q_emb], n_results=min(k, self.collection.count()))
        return results["documents"][0] if results["documents"] else []


# Initialize RAG and ingest a sample earnings call excerpt for demonstration
earnings_rag = EarningsCallRAG()

sample_transcript = """
NVIDIA Q4 FY25 Earnings Call Transcript Excerpt:
CEO Jensen Huang: Data center revenue reached a record $35.6 billion in Q4, up 93% year-over-year, driven by
strong demand for our Hopper and Blackwell computing platforms. Cloud service providers represented roughly
half of our Data Center revenue as hyperscalers deploy AI infrastructure at scale.
CFO Colette Kress: Gross margins for the fourth quarter were 73.5%. As Blackwell ramps in the first half of
the fiscal year, we expect gross margins to be in the low-70s range before returning to the mid-70s later in
the year. First quarter revenue is expected to be approximately $43.0 billion, plus or minus 2%.
"""

num_chunks = earnings_rag.ingest_text(sample_transcript, source_id="NVDA_Q4_Transcript")
print(f"✅ Ingested {num_chunks} transcript chunk(s) into ChromaDB.")


@tool
def search_earnings_call(query: str) -> str:
    """Search indexed earnings call transcripts for management commentary, guidance, and margins."""
    passages = earnings_rag.search(query, k=3)
    if not passages:
        return "No relevant passages found in earnings transcripts."
    return "\n---\n".join(passages)


print("\n🔍 RAG Query Result ('What is NVIDIA Q1 revenue guidance and gross margin outlook?'):")
print(search_earnings_call.invoke({"query": "What is NVIDIA Q1 revenue guidance and gross margin outlook?"}))

---
## Phase 3: Advanced Logic & Polish 

### Week 6: Reflexion (Self-Correction & Hallucination Guard)
Implementing a **Reflexion** loop where a Critic agent inspects the Analyst's draft response against raw tool outputs to catch unsupported claims, fabricated numbers, or missing risk disclosures, forcing iterative revisions until verified.

In [ ]:
class ReflexionFinancialAgent:
    """Self-correcting financial analyst agent using Draft -> Critique -> Revise cycles."""

    def __init__(self, llm: Optional[ChatGroq] = None, max_iterations: int = 2):
        self.llm = llm or get_llm()
        self.max_iterations = max_iterations

    def generate_draft(self, query: str, context_data: str) -> str:
        prompt = (
            f"SOURCE DATA:\n{context_data}\n\n"
            f"QUESTION: {query}\n\n"
            "Write a detailed financial analysis using ONLY the facts and numbers in SOURCE DATA."
        )
        return self.llm.invoke([
            SystemMessage(content="You are a precise financial analyst."),
            HumanMessage(content=prompt)
        ]).content

    def critique(self, query: str, context_data: str, draft: str) -> Dict[str, Any]:
        prompt = (
            f"SOURCE DATA:\n{context_data}\n\n"
            f"ANALYST DRAFT:\n{draft}\n\n"
            "Audit the draft against the source data. Check for:\n"
            "1. Any numbers or metrics not present in SOURCE DATA (hallucinations).\n"
            "2. Logical contradictions or missing key context from SOURCE DATA.\n"
            "If the draft is 100% faithful and complete, start your response with 'APPROVED'. "
            "Otherwise, start with 'REVISION_NEEDED' and list the exact corrections required."
        )
        res = self.llm.invoke([
            SystemMessage(content="You are a strict Financial Compliance & Fact-Checking Auditor."),
            HumanMessage(content=prompt)
        ]).content
        approved = res.strip().upper().startswith("APPROVED")
        return {"approved": approved, "critique": res}

    def revise(self, query: str, context_data: str, draft: str, critique_text: str) -> str:
        prompt = (
            f"SOURCE DATA:\n{context_data}\n\n"
            f"PREVIOUS DRAFT:\n{draft}\n\n"
            f"AUDITOR CRITIQUE:\n{critique_text}\n\n"
            f"Rewrite the analysis for '{query}', fixing every issue flagged by the auditor while relying strictly on SOURCE DATA."
        )
        return self.llm.invoke([
            SystemMessage(content="You are a senior financial analyst revising a report after compliance audit."),
            HumanMessage(content=prompt)
        ]).content

    def run(self, query: str, context_data: str) -> Dict[str, Any]:
        draft = self.generate_draft(query, context_data)
        history = [{"step": "initial_draft", "content": draft}]

        for i in range(1, self.max_iterations + 1):
            audit = self.critique(query, context_data, draft)
            history.append({"step": f"critique_{i}", "approved": audit["approved"], "content": audit["critique"]})
            if audit["approved"]:
                break
            draft = self.revise(query, context_data, draft, audit["critique"])
            history.append({"step": f"revision_{i}", "content": draft})

        return {"final_answer": draft, "iterations": len(history), "trace": history}


# Test Reflexion Agent
raw_context = (
    get_stock_price.invoke({"ticker": "AAPL"}) + "\n\n" +
    get_stock_history_tool.invoke({"ticker": "AAPL", "days": 30})
)
reflexion_agent = ReflexionFinancialAgent(max_iterations=2)
reflexion_result = reflexion_agent.run("Evaluate Apple's current valuation and 30-day momentum.", raw_context)

print(f"✅ Completed in {reflexion_result['iterations']} steps.")
print("\n--- FINAL VERIFIED REPORT ---")
print(reflexion_result["final_answer"])

### Week 7: Backtesting & Time Travel Engine
Evaluating agent decision quality without look-ahead bias by restricting `MarketData` to a historical `as_of_date`, generating a `BUY` / `HOLD` / `SELL` call using only data available on that date, and comparing against the actual subsequent 30-day return.

In [ ]:
class BacktestingEngine:
    """Time-travel backtesting engine to evaluate AI analyst recommendations on historical dates."""

    def __init__(self, market: MarketData, llm: Optional[ChatGroq] = None):
        self.market = market
        self.llm = llm or get_llm()

    def run_backtest(self, ticker: str, as_of_date: str, forward_days: int = 30) -> Dict[str, Any]:
        """Run agent analysis strictly using data up to `as_of_date`, then compare with actual forward return."""
        # 1. Historical data strictly prior to as_of_date (no look-ahead bias)
        past_df = self.market.get_stock_history(ticker, days=60, as_of_date=as_of_date)
        price_at_decision = float(past_df["Close"].iloc[-1])
        price_60d_ago = float(past_df["Close"].iloc[0])
        prior_return_pct = ((price_at_decision - price_60d_ago) / price_60d_ago) * 100
        sma_20 = float(past_df["Close"].tail(20).mean())

        # 2. Future window to measure actual outcome
        future_end_dt = datetime.strptime(as_of_date, "%Y-%m-%d") + timedelta(days=forward_days + 10)
        future_df = self.market.get_stock_history(
            ticker,
            days=forward_days + 10,
            as_of_date=future_end_dt.strftime("%Y-%m-%d")
        )
        post_df = future_df[future_df.index >= as_of_date]
        price_after = float(post_df["Close"].iloc[min(len(post_df) - 1, forward_days)])
        actual_return_pct = ((price_after - price_at_decision) / price_at_decision) * 100

        # 3. Ask LLM for decision as of `as_of_date`
        snapshot = (
            f"Date: {as_of_date} | Ticker: {ticker.upper()}\n"
            f"Close Price: ${price_at_decision:.2f} | 20-Day SMA: ${sma_20:.2f}\n"
            f"Prior 60-Day Trend: {prior_return_pct:+.2f}%\n"
            f"60-Day High: ${past_df['High'].max():.2f} | 60-Day Low: ${past_df['Low'].min():.2f}"
        )
        prompt = (
            f"{snapshot}\n\n"
            "Based ONLY on this historical technical snapshot, issue a single recommendation word "
            "(BUY, HOLD, or SELL) on the first line, followed by a one-sentence rationale."
        )
        reply = self.llm.invoke([
            SystemMessage(content="You are a quantitative swing trader backtesting historical signals."),
            HumanMessage(content=prompt)
        ]).content.strip()

        first_line = reply.splitlines()[0].upper()
        action = next((act for act in ("BUY", "SELL", "HOLD") if act in first_line), "HOLD")

        is_correct = (
            (action == "BUY" and actual_return_pct > 0) or
            (action == "SELL" and actual_return_pct < 0) or
            (action == "HOLD" and abs(actual_return_pct) <= 5.0)
        )

        return {
            "ticker": ticker.upper(),
            "as_of_date": as_of_date,
            "entry_price": round(price_at_decision, 2),
            "exit_price": round(price_after, 2),
            "recommendation": action,
            "actual_return_pct": round(actual_return_pct, 2),
            "profitable_call": is_correct,
            "rationale": reply.splitlines()[-1],
        }


backtester = BacktestingEngine(market_data)
bt_results = [
    backtester.run_backtest("AAPL", "2024-05-01", forward_days=30),
    backtester.run_backtest("NVDA", "2024-05-01", forward_days=30),
    backtester.run_backtest("MSFT", "2024-05-01", forward_days=30),
]

bt_df = pd.DataFrame(bt_results)
display_cols = ["ticker", "as_of_date", "entry_price", "exit_price", "recommendation", "actual_return_pct", "profitable_call"]
print("📈 Historical Backtest Results (30-Day Forward Horizon):")
print(bt_df[display_cols].to_string(index=False))

### Week 8: Streamlit Web Application & Deployment
The complete web interface is packaged in `app.py` alongside `requirements.txt` and `README.md` for one-click deployment to **Hugging Face Spaces** or local execution via Streamlit.

#### Running the Streamlit App Locally
```bash
streamlit run app.py
```

---
> ⚠️ **Disclaimer**: This project is for educational and research purposes only. Nothing generated by this system constitutes financial or investment advice.